# 01 - Preprocessing

Creates the canonical `data/preprocessed.csv` used by tabular, temporal, graph, and fusion notebooks.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

RAW_PATH = "../data/fraud_oracle_with_telematics.csv"
OUT_PATH = "../data/preprocessed.csv"

df = pd.read_csv(RAW_PATH)
print("Raw shape:", df.shape)
df.head()

Raw shape: (15420, 43)


,Month,WeekOfMonth,DayOfWeek,Make,AccidentArea,DayOfWeekClaimed,MonthClaimed,WeekOfMonthClaimed,Sex,MaritalStatus,...,avg_speed_kmph,max_speed_kmph,hard_brakes_per_trip,rapid_acceleration_events,trip_duration_minutes,distance_km,night_driving_ratio,urban_driving_ratio,harsh_cornering_events,idle_time_minutes
0,Dec,5,Wednesday,Honda,Urban,Tuesday,Jan,1,Female,Single,...,49.967142,73.579620,3,4,63.301642,52.716702,0.145654,0.774319,1,5.252264
1,Jan,3,Wednesday,Honda,Urban,Monday,Jan,4,Male,Single,...,43.617357,62.034068,5,7,27.244272,19.805386,0.071297,0.960082,2,7.935391
2,Oct,5,Friday,Honda,Urban,Thursday,Nov,2,Male,Married,...,51.476885,78.446508,4,4,13.644782,11.706515,0.649385,0.954290,1,1.546263
3,Jun,2,Saturday,Toyota,Rural,Friday,Jul,1,Male,Married,...,60.230299,76.137923,1,8,24.125514,24.218115,0.042510,0.961001,3,0.542277
4,Jan,5,Monday,Honda,Urban,Tuesday,Feb,2,Female,Single,...,42.658466,63.505354,3,2,50.189848,35.683699,0.433904,0.908334,0,4.501436


In [2]:
# Normalize column names and text values.
df.columns = df.columns.str.strip().str.lower()

for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype(str).str.strip().str.lower()

# Keep a stable key for branch fusion. PolicyNumber is an identifier, not a model feature.
if "policynumber" in df.columns:
    df.insert(0, "claim_id", df["policynumber"].astype(int))
else:
    df.insert(0, "claim_id", np.arange(len(df)))

# Remove leakage/administrative identifiers from the model feature set.
for col in ["policynumber", "repnumber"]:
    if col in df.columns:
        df = df.drop(columns=col)

print("After ID handling:", df.shape)

After ID handling: (15420, 42)


In [3]:
# Missing values.
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
df[num_cols] = df[num_cols].apply(lambda s: s.fillna(s.median()))

cat_cols = df.select_dtypes(include=["object", "string"]).columns
for col in cat_cols:
    mode = df[col].mode(dropna=True)
    df[col] = df[col].fillna(mode.iloc[0] if len(mode) else "unknown")

print("Missing values:", int(df.isna().sum().sum()))

Missing values: 0


In [4]:
# Convert range-like insurance columns to numeric midpoints.
range_mappings = {
    "days_policy_claim": {"none": 0, "1 to 7": 3, "8 to 15": 10, "15 to 30": 20, "more than 30": 40},
    "days_policy_accident": {"none": 0, "1 to 7": 3, "8 to 15": 10, "15 to 30": 20, "more than 30": 40},
    "pastnumberofclaims": {"none": 0, "1": 1, "2 to 4": 3, "more than 4": 5},
    "numberofsuppliments": {"none": 0, "1 to 2": 1.5, "3 to 5": 4, "more than 5": 6},
}

for col, mapping in range_mappings.items():
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].replace(mapping), errors="coerce").fillna(0)

In [5]:
# Telematics and claim-risk features.
if {"max_speed_kmph", "avg_speed_kmph"}.issubset(df.columns):
    df["speeding_risk"] = (df["max_speed_kmph"] > 120).astype(int)
    df["speed_volatility"] = (df["max_speed_kmph"] - df["avg_speed_kmph"]).abs()

if "hard_brakes_per_trip" in df.columns:
    df["harsh_braking_risk"] = (df["hard_brakes_per_trip"] > 5).astype(int)
if "rapid_acceleration_events" in df.columns:
    df["harsh_acceleration_risk"] = (df["rapid_acceleration_events"] > 5).astype(int)
if "harsh_cornering_events" in df.columns:
    df["harsh_cornering_risk"] = (df["harsh_cornering_events"] > 5).astype(int)

behavior_cols = ["hard_brakes_per_trip", "rapid_acceleration_events", "harsh_cornering_events"]
if set(behavior_cols).issubset(df.columns):
    df["harsh_driving_index"] = df[behavior_cols].mean(axis=1)

if "night_driving_ratio" in df.columns:
    df["high_night_driving"] = (df["night_driving_ratio"] > 0.5).astype(int)
if "urban_driving_ratio" in df.columns:
    df["high_urban_driving"] = (df["urban_driving_ratio"] > 0.7).astype(int)
if "days_policy_claim" in df.columns:
    df["fast_claim"] = (df["days_policy_claim"] < 7).astype(int)
if "pastnumberofclaims" in df.columns:
    df["high_claim_history"] = (df["pastnumberofclaims"] > 2).astype(int)
if {"fast_claim", "harsh_driving_index"}.issubset(df.columns):
    df["claim_driving_risk"] = df["fast_claim"] * df["harsh_driving_index"]

In [6]:
# Binary and categorical encoding.
for col in ["policereportfiled", "witnesspresent"]:
    if col in df.columns:
        df[col] = df[col].map({"yes": 1, "no": 0}).fillna(0).astype(int)

cat_cols = [col for col in df.select_dtypes(include=["object", "string"]).columns if col != "fraudfound_p"]
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

print("Encoded categorical columns:", len(cat_cols))

Encoded categorical columns: 18


In [7]:
# Scale numeric features while preserving IDs, target, and year.
target_col = "fraudfound_p"
if target_col not in df.columns:
    raise ValueError("Expected target column fraudfound_p was not found")

exclude_from_scaling = {"claim_id", target_col, "year"}
scale_cols = [c for c in df.select_dtypes(include=["int64", "float64"]).columns if c not in exclude_from_scaling]

scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

df[target_col] = df[target_col].astype(int)
df["claim_id"] = df["claim_id"].astype(int)
df = df.sort_values("claim_id").reset_index(drop=True)

In [8]:
# Final validation and save.
assert df["claim_id"].is_unique, "claim_id must be unique for fusion"
assert df.isna().sum().sum() == 0, "Preprocessed data still has missing values"

df.to_csv(OUT_PATH, index=False)

print("Saved:", OUT_PATH)
print("Shape:", df.shape)
print("Fraud distribution:")
print(df["fraudfound_p"].value_counts())
df.head()

Saved: ../data/preprocessed.csv
Shape: (15420, 53)
Fraud distribution:
fraudfound_p
0    14497
1      923
Name: count, dtype: int64


,claim_id,month,weekofmonth,dayofweek,make,accidentarea,dayofweekclaimed,monthclaimed,weekofmonthclaimed,sex,...,speed_volatility,harsh_braking_risk,harsh_acceleration_risk,harsh_cornering_risk,harsh_driving_index,high_night_driving,high_urban_driving,fast_claim,high_claim_history,claim_driving_risk
0,1,-1.035963,1.717545,1.500542,-0.778873,0.340019,0.790376,-0.467838,-1.345408,-2.317736,...,0.725376,-0.301255,-0.524215,-0.134501,-0.340056,-0.990062,1.139325,-0.008053,-0.972492,-0.008053
1,2,-0.449364,0.164199,1.500542,-0.778873,0.340019,-0.968740,-0.467838,1.037295,0.431455,...,-0.317649,-0.301255,1.907613,-0.134501,1.661724,-0.990062,1.139325,-0.008053,-0.972492,-0.008053
2,3,1.310432,1.717545,-1.418572,-0.778873,0.340019,0.350597,0.998077,-0.551174,0.431455,...,1.399306,-0.301255,-0.524215,-0.134501,-0.006426,1.010037,1.139325,-0.008053,-0.972492,-0.008053
3,4,0.137234,-0.612473,-0.445534,1.303376,-2.941014,-1.408519,-0.174655,-1.345408,0.431455,...,-0.821335,-0.301255,1.907613,-0.134501,0.994464,-0.990062,1.139325,-0.008053,-0.972492,-0.008053
4,5,-0.449364,1.717545,-0.932053,-0.778873,0.340019,0.790376,-0.761021,-0.551174,-2.317736,...,0.170197,-0.301255,-0.524215,-0.134501,-1.340946,-0.990062,1.139325,-0.008053,-0.972492,-0.008053
